In [1]:
# Step 1: Import libraries

import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import os
from tqdm import tqdm

In [2]:
# Step 2: Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Step 3: Define project paths

BASE_DIR = "/content/drive/MyDrive/gpt-from-scratch"
DATA_DIR = os.path.join(BASE_DIR, "data")
SRC_DIR = os.path.join(BASE_DIR, "src")
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
NOTEBOOKS_DIR = os.path.join(BASE_DIR, "notebooks")

for folder in [BASE_DIR, DATA_DIR, SRC_DIR, CKPT_DIR, RESULTS_DIR, NOTEBOOKS_DIR]:
    os.makedirs(folder, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("Folders are ready.")
print("DATA_DIR:", DATA_DIR)
print("SRC_DIR:", SRC_DIR)
print("CKPT_DIR:", CKPT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

BASE_DIR: /content/drive/MyDrive/gpt-from-scratch
Folders are ready.
DATA_DIR: /content/drive/MyDrive/gpt-from-scratch/data
SRC_DIR: /content/drive/MyDrive/gpt-from-scratch/src
CKPT_DIR: /content/drive/MyDrive/gpt-from-scratch/checkpoints
RESULTS_DIR: /content/drive/MyDrive/gpt-from-scratch/results


In [4]:
# Step 4: Inspect project files

for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

gpt-from-scratch/
  requirements.txt
  gpt_marketing_scripts.json
  data/
    pretrain_corpus.txt
    tokenizer_vocab.json
    pretrain.txt
    sft.jsonl
  src/
    model.py
    tokenizer.py
    __pycache__/
      tokenizer.cpython-312.pyc
      model.cpython-312.pyc
  checkpoint/
  results/
    checkpoint-9/
      config.json
      generation_config.json
      model.safetensors
      training_args.bin
      optimizer.pt
      trainer_state.json
      rng_state.pth
      scheduler.pt
  demo/
  notebooks/
  checkpoints/
    tokenizer/
      bpe_tokenizer.json


In [5]:
# Step 5: Check important files

important_files = [
    os.path.join(BASE_DIR, "gpt_marketing_scripts.json"),
    os.path.join(DATA_DIR, "pretrain.txt"),
    os.path.join(DATA_DIR, "pretrain_corpus.txt"),
    os.path.join(DATA_DIR, "sft.jsonl"),
]

for file_path in important_files:
    print(file_path, "->", os.path.exists(file_path))

/content/drive/MyDrive/gpt-from-scratch/gpt_marketing_scripts.json -> True
/content/drive/MyDrive/gpt-from-scratch/data/pretrain.txt -> True
/content/drive/MyDrive/gpt-from-scratch/data/pretrain_corpus.txt -> True
/content/drive/MyDrive/gpt-from-scratch/data/sft.jsonl -> True


In [6]:
 #Step 6: Convert marketing scripts JSON to SFT dataset

input_path = os.path.join(BASE_DIR, "gpt_marketing_scripts.json")
output_path = os.path.join(DATA_DIR, "sft.jsonl")

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

samples = []

for category, items in data.items():
    for item in items:
        instruction = item.get("instruction", "")
        response = item.get("output", "")

        text = f"### Instruction:\n{instruction}\n\n### Response:\n{response}"

        samples.append({"text": text})

with open(output_path, "w", encoding="utf-8") as f:
    for s in samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

print("SFT dataset created!")
print("Total samples:", len(samples))
print("Saved to:", output_path)

SFT dataset created!
Total samples: 6
Saved to: /content/drive/MyDrive/gpt-from-scratch/data/sft.jsonl


In [7]:
# Step 7: Install tokenizer library

!pip install -q tokenizers

In [8]:
# Step 8: Train a BPE tokenizer on pretraining + SFT data

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.processors import ByteLevel as ByteLevelProcessor

tokenizer_dir = os.path.join(CKPT_DIR, "tokenizer")
os.makedirs(tokenizer_dir, exist_ok=True)

tokenizer_path = os.path.join(tokenizer_dir, "bpe_tokenizer.json")

training_files = [
    os.path.join(DATA_DIR, "pretrain.txt"),
    os.path.join(DATA_DIR, "sft.jsonl"),
]

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()
tokenizer.post_processor = ByteLevelProcessor(trim_offsets=True)

trainer = BpeTrainer(
    vocab_size=3000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)

tokenizer.train(training_files, trainer)
tokenizer.save(tokenizer_path)

print("Tokenizer trained successfully!")
print("Saved to:", tokenizer_path)
print("Vocab size:", tokenizer.get_vocab_size())

Tokenizer trained successfully!
Saved to: /content/drive/MyDrive/gpt-from-scratch/checkpoints/tokenizer/bpe_tokenizer.json
Vocab size: 412


In [9]:
# Step 9: Test tokenizer

test_text = "اكتب إعلانًا قصيرًا عن كفالة يتيم بأسلوب مؤثر"

encoded = tokenizer.encode(test_text)

print("Original text:")
print(test_text)
print("\nToken IDs:")
print(encoded.ids[:50])
print("\nTokens:")
print(encoded.tokens[:50])
print("\nDecoded:")
print(tokenizer.decode(encoded.ids))

Original text:
اكتب إعلانًا قصيرًا عن كفالة يتيم بأسلوب مؤثر

Token IDs:
[92, 191, 229, 246, 123, 152, 92, 153, 122, 265, 152, 92, 288, 362, 296, 131, 127, 111, 93, 104, 110, 106, 304, 207, 98]

Tokens:
['Ø§', 'ÙĥØªØ¨', 'ĠØ¥', 'Ø¹ÙĦ', 'Ø§ÙĨ', 'Ùĭ', 'Ø§', 'ĠÙĤ', 'Øµ', 'ÙĬØ±', 'Ùĭ', 'Ø§', 'ĠØ¹ÙĨ', 'ĠÙĥÙģØ§ÙĦØ©', 'ĠÙĬØªÙĬÙħ', 'ĠØ¨', 'Ø£', 'Ø³', 'ÙĦ', 'ÙĪ', 'Ø¨', 'ĠÙħ', 'Ø¤', 'Ø«', 'Ø±']

Decoded:
اكتب إعلانًا قصيرًا عن كفالة يتيم بأسلوب مؤثر


In [10]:
# Step 10: Load pretraining data

pretrain_path = os.path.join(DATA_DIR, "pretrain.txt")

with open(pretrain_path, "r", encoding="utf-8") as f:
    pretrain_text = f.read()

print("Length of text:", len(pretrain_text))
print("\nSample text:")
print(pretrain_text[:500])

Length of text: 416

Sample text:
السلام عليكم. هذا نص تجريبي طويل لتدريب GPT. كرر هذا النص 1000 مرة لزيادة الحجم...
#  (ماذا لو صادف عطاؤك ليلةَ القدر؟
وماذا لو كان عطاؤك سقيا الماء التي أوصى بها رسول الله ﷺ؟
قال ﷺ: «أفضلُ الصدقةِ سقيُ الماء».
وليلةُ القدر خيرٌ من ألفِ شهر، والأجرُ فيها مضاعف.
بـ ٩٦ ريالًا فقط، يتكفّل متجر غيث بإخراج كراتين ماءٍ عنك خلال العشر الأواخر من رمضان
لعلّها تُصادف ليلةَ القدر.
اغتنم الفرصة، وارفع الشاشة الآن) حتى >2MB



In [11]:
# Step 11: Tokenize the pretraining text

encoded = tokenizer.encode(pretrain_text)

token_ids = encoded.ids

print("Total tokens:", len(token_ids))
print("First 50 tokens:")
print(token_ids[:50])

Total tokens: 199
First 50 tokens:
[325, 93, 311, 289, 340, 10, 292, 316, 122, 124, 220, 321, 96, 395, 320, 115, 103, 147, 321, 69, 17, 20, 22, 10, 119, 98, 98, 292, 101, 97, 122, 69, 12, 301, 11, 338, 115, 305, 148, 334, 101, 145, 109, 100, 134, 68, 6, 69, 69, 8]


In [12]:
# Step 12: Convert tokens to tensor

data = torch.tensor(token_ids, dtype=torch.long)

print("Tensor shape:", data.shape)

Tensor shape: torch.Size([199])


In [13]:
# Step 13: Training parameters

batch_size = 8
block_size = 32

In [14]:
# Step 14: Create batch loader

def get_batch():

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

In [15]:
# Step 15: Test batch

xb, yb = get_batch()

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)

Input shape: torch.Size([8, 32])
Target shape: torch.Size([8, 32])


In [16]:
# Step 16: Model hyperparameters

vocab_size = tokenizer.get_vocab_size()

n_embed = 128
n_head = 4
n_layer = 4
dropout = 0.1

In [17]:
# Step 17: Token and Position Embeddings

token_embedding_table = nn.Embedding(vocab_size, n_embed)
position_embedding_table = nn.Embedding(block_size, n_embed)

print("Vocab size:", vocab_size)
print("Embedding size:", n_embed)

Vocab size: 412
Embedding size: 128


In [18]:
# Step 18: Test embeddings

xb, yb = get_batch()

token_embeddings = token_embedding_table(xb)
position_embeddings = position_embedding_table(torch.arange(block_size))

x = token_embeddings + position_embeddings

print("Embedding output shape:", x.shape)

Embedding output shape: torch.Size([8, 32, 128])


In [19]:
# Step 19: Self-Attention Head

class Head(nn.Module):
    def ــinitــ(self, head_size):
        super().ــinitــ()

        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)

        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v

        return out

In [20]:
# Step 20: Test attention head

# Redefining Head class with corrected __init__ to fix the error
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)

        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v

        return out

head_size = n_embed // n_head
head = Head(head_size)

out = head(x)

print("Attention output shape:", out.shape)

Attention output shape: torch.Size([8, 32, 32])


In [21]:
# Step 21: Multi-Head Attention

class MultiHeadAttention(nn.Module):
    def _init_(self, num_heads, head_size):
        super()._init_()

        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [22]:
# Step 22: Test multi-head attention

# Redefine MultiHeadAttention with corrected __init__ method
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

head_size = n_embed // n_head
mha = MultiHeadAttention(n_head, head_size)

out = mha(x)

print("Multi-head output shape:", out.shape)

Multi-head output shape: torch.Size([8, 32, 128])


In [23]:
# Step 23: FeedForward network

class FeedForward(nn.Module):
    def _init_(self, n_embed):
        super()._init_()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [24]:
# Step 24: Test feedforward

# Redefine FeedForward with corrected __init__ method
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

ff = FeedForward(n_embed)

out = ff(x)

print("FeedForward output shape:", out.shape)

FeedForward output shape: torch.Size([8, 32, 128])


In [25]:
# Step 25: Transformer Block

class Block(nn.Module):
    def _init_(self, n_embed, n_head):
        super()._init_()

        head_size = n_embed // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)

        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [26]:
# Step 26: Test transformer block

# Redefine Block with corrected __init__ method
class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()

        head_size = n_embed // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)

        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

block = Block(n_embed, n_head)

out = block(x)

print("Block output shape:", out.shape)

Block output shape: torch.Size([8, 32, 128])


In [27]:
# Step 27: GPT Model

class GPTLanguageModel(nn.Module):

    def _init_(self):
        super()._init_()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)

        self.blocks = nn.Sequential(
            *[Block(n_embed, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))

        x = tok_emb + pos_emb
        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [28]:
# Step 28: Create model

# Redefine GPTLanguageModel with corrected __init__ method
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)

        self.blocks = nn.Sequential(
            *[Block(n_embed, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))

        x = tok_emb + pos_emb
        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

model = GPTLanguageModel()

logits, loss = model(xb, yb)

print("Logits shape:", logits.shape)
print("Loss:", loss)

Logits shape: torch.Size([256, 412])
Loss: tensor(6.2459, grad_fn=<NllLossBackward0>)


In [29]:
# Step 29: Set device

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = GPTLanguageModel().to(device)

Using device: cuda


In [30]:
# Step 30: Update get_batch for device

def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [31]:
# Step 31: Optimizer

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("Optimizer ready")

Optimizer ready


In [32]:
# Step 32: Training loop

# Redefine GPTLanguageModel with the fix for device placement of position embeddings
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)

        self.blocks = nn.Sequential(
            *[Block(n_embed, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        # Fix: Move position indices to the same device as the model
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))

        x = tok_emb + pos_emb
        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

# Re-instantiate the model with the corrected class definition and move to device
model = GPTLanguageModel().to(device)

# Re-initialize the optimizer as the model parameters have changed
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

max_iters = 300
eval_interval = 50

for iter in range(max_iters):

    xb, yb = get_batch()

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % eval_interval == 0:
        print(f"step {iter}: loss {loss.item():.4f}")

step 0: loss 6.1873
step 50: loss 0.6945
step 100: loss 0.1776
step 150: loss 0.0985
step 200: loss 0.0507
step 250: loss 0.0438


In [33]:
# Step 33: Redefine model with generate()

class GPTLanguageModel(nn.Module):

    def _init_(self):
        super()._init_()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)

        self.blocks = nn.Sequential(
            *[Block(n_embed, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [34]:
# Step 34: Recreate model and quick retrain

# Redefine model with generate() and corrected __init__ method
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)

        self.blocks = nn.Sequential(
            *[Block(n_embed, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

max_iters = 200

for iter in range(max_iters):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 50 == 0:
        print(f"step {iter}: loss {loss.item():.4f}")

step 0: loss 6.2188
step 50: loss 0.9729
step 100: loss 0.2617
step 150: loss 0.0893


In [35]:
# Step 35: Simple encode/decode helpers

def encode_text(text):
    return tokenizer.encode(text).ids

def decode_tokens(tokens):
    return tokenizer.decode(tokens)

In [36]:
# Step 36: Generate text from a prompt

prompt = "اكتب إعلانًا عن كفالة يتيم"

start_ids = encode_text(prompt)
x = torch.tensor([start_ids], dtype=torch.long, device=device)

generated_ids = model.generate(x, max_new_tokens=40)[0].tolist()
generated_text = decode_tokens(generated_ids)

print("PROMPT:")
print(prompt)
print("\nGENERATED TEXT:")
print(generated_text)

PROMPT:
اكتب إعلانًا عن كفالة يتيم

GENERATED TEXT:
اكتب إعلانًا عن كفالة يتيم�٦ ريالًا فقط، يتكفّها رسول الله ﷺ: «أفضلُ الصدقةِ سقيُ الماء».
وليلةُ القدر


In [37]:
# Step 37: Save model

torch.save(model.state_dict(), "gpt_arabic_model.pt")

print("Model saved successfully")

Model saved successfully


In [38]:
# Step 38: Convert JSONL dataset to training text

import json

input_file = "nonprofit_marketing_dataset_clean_200.jsonl"
output_file = "sft_dataset.txt"

count = 0

with open(input_file, "r", encoding="utf-8") as f, open(output_file, "w", encoding="utf-8") as out:
    for line in f:
        data = json.loads(line)

        instruction = data["instruction"]
        response = data["output"]

        text = f"""### Instruction:
{instruction}

### Response:
{response}

"""

        out.write(text)
        count += 1

print("Dataset converted successfully")
print("Examples:", count)

Dataset converted successfully
Examples: 200


In [39]:
# Step 39: Preview dataset

with open("sft_dataset.txt","r",encoding="utf-8") as f:
    print(f.read(1000))

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال العلاج.

### Response:
الخير يبدأ بخطوة بسيطة منك.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال السلال الغذائية.

### Response:
لا تؤجل الخير.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال العلاج.

### Response:
هناك من ينتظر مساعدتك اليوم.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال الأيتام.

### Response:
لا تؤجل الخير.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال كسوة الشتاء.

### Response:
كل ريال قد يغير حياة شخص.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال التبرع العام.

### Response:
كن سبب الفرج لشخص محتاج.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال سقيا الماء.

### Response:
كن سبب الفرج لشخص محتاج.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال ترميم المنازل.

### Response:
عطاؤك قد يصنع قصة أمل جديدة.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال ترميم المنازل.

### Response:
تبرعك اليوم قد يكون سبب ابتسامة طفل غداً.

### Instruction:
اكتب هوك تسويقي ل

In [40]:
# Step 40: Load SFT dataset

with open("sft_dataset.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Dataset length:", len(text))

Dataset length: 33579


In [41]:
# Step 41: Encode SFT dataset

data = torch.tensor(encode_text(text), dtype=torch.long)

print("Total tokens:", len(data))

Total tokens: 18963


In [42]:
# SFT Step A: Load converted SFT text

with open("sft_dataset.txt", "r", encoding="utf-8") as f:
    sft_text = f.read()

print("SFT text length:", len(sft_text))
print(sft_text[:500])

SFT text length: 33579
### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال العلاج.

### Response:
الخير يبدأ بخطوة بسيطة منك.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال السلال الغذائية.

### Response:
لا تؤجل الخير.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال العلاج.

### Response:
هناك من ينتظر مساعدتك اليوم.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال الأيتام.

### Response:
لا تؤجل الخير.

### Instruction:
اكتب هوك تسويقي لجمعية تعمل في مجال كسوة الشتاء.

### Response:
كل ريال قد يغي


In [43]:
# SFT Step B: Encode SFT text

sft_ids = encode_text(sft_text)
print("Total SFT token ids:", len(sft_ids))
print("Max token id:", max(sft_ids))
print("Tokenizer vocab size:", tokenizer.get_vocab_size())

Total SFT token ids: 18963
Max token id: 411
Tokenizer vocab size: 412


In [44]:
# SFT Step C: Convert to tensor

sft_data = torch.tensor(sft_ids, dtype=torch.long)
print("SFT tensor shape:", sft_data.shape)

SFT tensor shape: torch.Size([18963])


In [45]:
# SFT Step D: Recreate model cleanly

vocab_size = tokenizer.get_vocab_size()

model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("Model recreated successfully")
print("Vocab size:", vocab_size)

Model recreated successfully
Vocab size: 412


In [46]:
# SFT Step D: Recreate model cleanly

vocab_size = tokenizer.get_vocab_size()

# Redefine all model component classes to ensure they pick up current global hyperparameters
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(
            *[Block(n_embed, n_head) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("Model recreated successfully")
print("Vocab size:", vocab_size)


Model recreated successfully
Vocab size: 412


In [48]:
for iter in range(max_iters):

    xb, yb = get_batch()

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % eval_interval == 0:
        print(f"step {iter}: loss {loss.item():.4f}")

step 0: loss 6.2145
step 50: loss 2.4508
step 100: loss 1.3648
step 150: loss 1.0994


In [50]:
# Step 43: Save the SFT-trained model

torch.save(model.state_dict(), "gpt_arabic_model_sft.pt")
print("SFT model saved successfully")

SFT model saved successfully


In [51]:
# Step 44: Test the SFT-trained model

prompt = "اكتب إعلانًا لجمعية تكفل الأيتام"

start_ids = encode_text(prompt)
x = torch.tensor([start_ids], dtype=torch.long, device=device)

generated_ids = model.generate(x, max_new_tokens=60)[0].tolist()
generated_text = decode_tokens(generated_ids)

print("PROMPT:")
print(prompt)
print("\nGENERATED TEXT:")
print(generated_text)

PROMPT:
اكتب إعلانًا لجمعية تكفل الأيتام

GENERATED TEXT:
اكتب إعلانًا لجمعية تكفل الأيتام.

### Response:
لاكتب�لا تعمل الخير لجمعية تعم.

### Response:
كن س التبرعم، مشاريع الآن

### Response:
. مشخ


In [52]:
!pip install gradio

In [53]:
import gradio as gr
import torch

def generate_marketing_text(prompt):

    start_ids = encode_text(prompt)
    x = torch.tensor([start_ids], dtype=torch.long, device=device)

    generated_ids = model.generate(x, max_new_tokens=80)[0].tolist()
    generated_text = decode_tokens(generated_ids)

    return generated_text


demo = gr.Interface(
    fn=generate_marketing_text,
    inputs=gr.Textbox(
        lines=3,
        placeholder="مثال: اكتب إعلانًا لجمعية تكفل الأيتام"
    ),
    outputs=gr.Textbox(lines=10),
    title="Arabic Nonprofit Marketing Generator",
    description="نموذج يولد نصوص تسويقية للجمعيات الخيرية"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://91e3e35a061be4dfbc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [54]:
import gradio as gr
import torch

def generate_marketing_text(prompt):

    start_ids = encode_text(prompt)
    x = torch.tensor([start_ids], dtype=torch.long, device=device)

    generated_ids = model.generate(x, max_new_tokens=80)[0].tolist()
    generated_text = decode_tokens(generated_ids)

    return generated_text


demo = gr.Interface(
    fn=generate_marketing_text,
    inputs=gr.Textbox(
        lines=3,
        placeholder="مثال: اكتب إعلانًا لجمعية تكفل الأيتام"
    ),
    outputs=gr.Textbox(lines=10),
    title="Arabic Nonprofit Marketing Generator",
    description="نموذج يولد نصوص تسويقية للجمعيات الخيرية"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://58cca805949f51af30.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
